# Notebook 05 — A/B Test Dashboard Final
**Tujuan:** Layout satu halaman siap presentasi — KPI + Verdict GO/NO-GO + 4 chart utama.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import json
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from pathlib import Path

OUT = Path('../output')

BLUE    = '#2563EB'
LIGHT   = '#93C5FD'
LIGHTER = '#DBEAFE'
RED     = '#DC2626'
GREEN   = '#059669'
GRAY    = '#6B7280'
BG      = '#F8FAFC'
ALPHA   = 0.05

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': BG,
    'axes.facecolor': '#FFFFFF',
})

df      = pd.read_parquet(OUT / 'df_clean.parquet')
results = json.loads((OUT / 'test_results.json').read_text())
print('Data loaded.')

In [ ]:
# ── Pre-compute semua nilai ──────────────────────────────────────────────────
ctrl  = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

p_ctrl  = ctrl['converted'].mean()
p_treat = treat['converted'].mean()
n_ctrl, n_treat = len(ctrl), len(treat)
conv_ctrl  = ctrl['converted'].sum()
conv_treat = treat['converted'].sum()

count  = np.array([conv_treat, conv_ctrl])
nobs   = np.array([n_treat,    n_ctrl])
z_stat, p_value_conv = proportions_ztest(count, nobs, alternative='larger')

ci_ctrl  = proportion_confint(conv_ctrl,  n_ctrl,  alpha=ALPHA, method='wilson')
ci_treat = proportion_confint(conv_treat, n_treat, alpha=ALPHA, method='wilson')

boot_ci_lo = results['revenue_test']['bootstrap_ci_lower']
boot_ci_hi = results['revenue_test']['bootstrap_ci_upper']
boot_ci_str = f'[${boot_ci_lo:+.3f}, ${boot_ci_hi:+.3f}]'

conv_verdict = 'GO ✅'    if p_value_conv < ALPHA else 'NO-GO ❌'
rev_verdict  = results['revenue_test']['verdict']
overall_verdict = 'GO ✅' if (p_value_conv < ALPHA or results['revenue_test']['p_value'] < ALPHA) else 'NO-GO ❌'
verdict_color = GREEN if '✅' in overall_verdict else RED

# Bootstrap data
np.random.seed(42)
boot_diffs = np.array([
    np.mean(np.random.choice(treat['revenue'].values, n_treat, replace=True)) -
    np.mean(np.random.choice(ctrl['revenue'].values,  n_ctrl,  replace=True))
    for _ in range(5_000)  # 5k untuk speed di dashboard
])

# Power curve data
analysis   = NormalIndPower()
h_actual   = proportion_effectsize(p_treat, p_ctrl)
sample_sizes = np.arange(500, 100_001, 500)
powers_actual = [analysis.power(effect_size=abs(h_actual), nobs1=n, alpha=0.05, ratio=1.0)
                 for n in sample_sizes]
powers_small  = [analysis.power(effect_size=0.05, nobs1=n, alpha=0.05, ratio=1.0)
                 for n in sample_sizes]
n_80 = analysis.solve_power(effect_size=abs(h_actual), power=0.80, alpha=0.05, ratio=1.0)

print(f'Conv p-value : {p_value_conv:.4f} → {conv_verdict}')
print(f'Overall      : {overall_verdict}')

In [ ]:
# ── Build Dashboard ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 15), facecolor=BG)
fig.suptitle('A/B Test Analysis Dashboard — Landing Page Experiment',
             fontsize=20, fontweight='bold', y=0.98, color='#1E3A5F')
fig.text(0.5, 0.955, f'n={len(df):,} users  |  α=0.05  |  Bootstrap CI: 10,000 iterations  |  Kaggle A/B Dataset',
         ha='center', fontsize=11, color=GRAY)

gs = gridspec.GridSpec(3, 4, figure=fig,
    height_ratios=[0.15, 0.43, 0.42],
    hspace=0.48, wspace=0.38,
    top=0.93, bottom=0.05, left=0.05, right=0.97)

# ── ROW 0: KPI Cards ────────────────────────────────────────────────────────
conv_lift = (p_treat - p_ctrl) / p_ctrl * 100
kpis = [
    ('Conv Rate Lift',   f'{conv_lift:+.2f}%',   BLUE  if conv_lift > 0 else RED),
    ('p-value (Conv)',   f'{p_value_conv:.4f}',   GREEN if p_value_conv < ALPHA else RED),
    ('Bootstrap 95% CI', boot_ci_str,             BLUE),
    ('Overall Verdict',  overall_verdict,         verdict_color),
]
for col, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, col])
    ax.axis('off')
    rect = mpatches.FancyBboxPatch((0, 0), 1, 1,
        boxstyle='round,pad=0.05', facecolor='#FFFFFF',
        edgecolor=LIGHTER, linewidth=1.5,
        transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    fs = 14 if len(value) > 12 else 17
    ax.text(0.5, 0.65, value, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center',
            fontsize=10, color=GRAY, transform=ax.transAxes)

# ── ROW 1, col 0-1: Conversion Rate Bar + CI ────────────────────────────────
ax_conv = fig.add_subplot(gs[1, :2])
groups  = ['Control\n(old page)', 'Treatment\n(new page)']
rates   = [p_ctrl * 100, p_treat * 100]
errs_lo = [(p_ctrl - ci_ctrl[0]) * 100,  (p_treat - ci_treat[0]) * 100]
errs_hi = [(ci_ctrl[1] - p_ctrl) * 100,  (ci_treat[1] - p_treat) * 100]
bar_colors = [BLUE, GREEN if p_treat >= p_ctrl else RED]

bars = ax_conv.bar(groups, rates, color=bar_colors, width=0.4,
    yerr=[errs_lo, errs_hi], capsize=10,
    error_kw={'linewidth': 2, 'ecolor': GRAY})
ax_conv.set_ylabel('Conversion Rate (%)')
ax_conv.yaxis.set_major_formatter(mticker.PercentFormatter())
ax_conv.set_title(f'Conversion Rate per Grup (95% CI)\nZ-test p={p_value_conv:.4f} — {conv_verdict}',
                  fontweight='bold', fontsize=12)
for bar, rate in zip(bars, rates):
    ax_conv.text(bar.get_x() + bar.get_width()/2, rate + 0.05,
                 f'{rate:.2f}%', ha='center', fontweight='bold', fontsize=11)

# ── ROW 1, col 2-3: Revenue KDE ─────────────────────────────────────────────
ax_kde = fig.add_subplot(gs[1, 2:])
ctrl_conv  = ctrl[ctrl['converted']==1]['revenue']
treat_conv = treat[treat['converted']==1]['revenue']
ctrl_conv.plot.kde(ax=ax_kde,  color=BLUE, linewidth=2.5,
                   label=f'Control  μ=${ctrl_conv.mean():.2f}')
treat_conv.plot.kde(ax=ax_kde, color=RED,  linewidth=2.5,
                    label=f'Treatment μ=${treat_conv.mean():.2f}')
ax_kde.set_xlabel('Revenue per Converter (USD)')
ax_kde.set_title('Revenue Distribution (Converter Only)\n(T-test p={:.4f} — {})'.format(
    results['revenue_test']['p_value'], rev_verdict),
    fontweight='bold', fontsize=12)
ax_kde.legend(fontsize=10)
ax_kde.set_xlim(0)

# ── ROW 2, col 0-1: Bootstrap Distribution ──────────────────────────────────
ax_boot = fig.add_subplot(gs[2, :2])
ax_boot.hist(boot_diffs, bins=60, color=LIGHT, edgecolor='white', linewidth=0.3)
ax_boot.axvline(0, color=GRAY, linestyle='--', linewidth=2, label='H₀: diff=0')
ax_boot.axvspan(boot_ci_lo, boot_ci_hi, alpha=0.2, color=RED,
                label=f'95% CI: {boot_ci_str}')
ax_boot.axvline(boot_ci_lo, color=RED, linestyle='--', linewidth=1.5)
ax_boot.axvline(boot_ci_hi, color=RED, linestyle='--', linewidth=1.5)
ax_boot.axvline(boot_diffs.mean(), color=BLUE, linewidth=2.5,
                label=f'Mean: ${boot_diffs.mean():+.4f}')
ax_boot.set_xlabel('Mean Revenue Difference (USD)')
ax_boot.set_ylabel('Frekuensi')
ax_boot.set_title('Bootstrap Distribution — Revenue Difference\n(5,000 iterasi)', fontweight='bold', fontsize=12)
ax_boot.legend(fontsize=9)

# ── ROW 2, col 2-3: Power Curve ─────────────────────────────────────────────
ax_power = fig.add_subplot(gs[2, 2:])
ax_power.plot(sample_sizes / 1000, powers_actual, color=BLUE, linewidth=2.5,
              label=f'Effect aktual (h={abs(h_actual):.3f})')
ax_power.plot(sample_sizes / 1000, powers_small,  color=LIGHT, linewidth=2,
              linestyle='--', label='Kecil (h=0.05)')
ax_power.axhline(0.80, color='black', linestyle='--', linewidth=1.5, alpha=0.6, label='80% power')
ax_power.axvline(n_80 / 1000, color=BLUE, linestyle='-.', linewidth=1.5, alpha=0.8)
ax_power.text(n_80 / 1000 + 0.5, 0.3, f'n≈{n_80:,.0f}\nper grup', color=BLUE, fontsize=9)
ax_power.set_xlabel('Sample Size per Grup (ribu)')
ax_power.set_ylabel('Statistical Power')
ax_power.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax_power.set_title('Power Curve (α=0.05)', fontweight='bold', fontsize=12)
ax_power.legend(fontsize=9, loc='lower right')
ax_power.set_ylim(0, 1.05)
ax_power.set_xlim(0, 100)

# ── Footer ──────────────────────────────────────────────────────────────────
fig.text(0.5, 0.012,
    'Prepared by Ray  |  Tools: Python, SciPy, Statsmodels, Matplotlib  |  Source: Kaggle A/B Testing Dataset',
    ha='center', fontsize=8.5, color=GRAY)

plt.savefig(OUT / 'dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
print('Saved: output/dashboard.png')
plt.show()

## Executive Summary

### Keputusan Bisnis: NO-GO — Jangan Deploy New Page

| Metric | Control | Treatment | Verdict |
|---|---|---|---|
| Conversion Rate | ~12.0% | ~11.9% | Tidak signifikan (p > 0.05) |
| Revenue per User | ~baseline | ~baseline | Tidak signifikan |
| Bootstrap 95% CI | — | Mencakup 0 | Tidak ada bukti lift |
| Statistical Power | — | — | Test terpowered dengan baik |

### Interpretasi
- New page **tidak lebih baik** dari old page — tidak ada cukup bukti untuk mengklaim lift conversion.
- Dataset ini dirancang sebagai *null result* — hasil NO-GO adalah **benar secara statistik**, bukan kegagalan analisis.
- ~70% A/B test di industri gagal mengalahkan baseline — null result adalah informasi berharga: iterasi, jangan deploy.

### Next Steps
1. **Hypothesis reformulation:** Identifikasi elemen spesifik mana yang mungkin berkontribusi (CTA warna, headline, layout).
2. **Segmented analysis:** Uji apakah ada subgroup (device, region, new vs returning) di mana treatment unggul.
3. **Minimum detectable effect:** Jika target lift 1%, butuh n>15,000/grup — pastikan experiment cukup lama berjalan.

> **Relevansi Indonesia:** Tokopedia dengan >100 juta active user bisa mendeteksi lift 0.5% hanya dalam 2-3 hari. Seller lokal skala kecil butuh minimum 7 hari runtime. ~70% A/B test gagal beat baseline — null result tetap informasi berharga.
